# Logit lens -- collect the per-layer curves (Colab, GPU)

For every item and every checkpoint, read the model's answer lean after each of the 33 layer
outputs (embeddings + 32 layers) at the last position of the **generation prompt** (ends at
`<|assistant|>\n`) -- the exact prompt whose first generated token *is* the behaviour the eval
labelled. (v1 used the scoring prompt with `The answer is:` appended; that changes the decision
on close calls, so its final layer agreed with the generated answer only 82-88% of the time.)

    D[layer] = logit(first token of context answer) - logit(first token of parametric answer)

`D > 0` = leaning to the context answer, `D < 0` = leaning to the model's own answer.
This equals log p(context tok) / p(parametric tok), because the softmax normaliser cancels.
After a newline the answer starts without a leading space, so its first token is the
no-space piece ("Dur", not "_Dur") -- the sanity cell checks this against real generations.

Runs every item (412) on every checkpoint listed in `CHECKPOINTS`, so the analysis notebook
(`logit_lens_analysis.ipynb`, local) can define flip/stable groups for any seed by itself.
One JSON per checkpoint is written to Drive; re-running skips finished checkpoints.

In [ ]:
import os

GITHUB_REPO_URL = "https://github.com/GIRIAYUSH/context-parametric-inversion-research.git"
REPO_DIR = "/content/context-parametric-inversion-research"
BRANCH = "dev"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}
!git log --oneline -3

!pip install -q peft accelerate

from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/CPI-Analysis-Results/logit-lens-v2"   # persistent; v2 = generation prompt
os.makedirs(OUT_DIR, exist_ok=True)
assert os.access(OUT_DIR, os.W_OK), "Drive output dir not writable"
print("writing to", OUT_DIR)

/content/context-parametric-inversion-research
From https://github.com/GIRIAYUSH/context-parametric-inversion-research
 * branch            dev        -> FETCH_HEAD
HEAD is now at 435ff3c flip set items
435ff3c (HEAD -> dev, origin/dev) flip set items
02b24cf Merge remote-tracking branch 'origin/dev' into dev
49e9806 Add no-context parametric-strength scoring notebook
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
writing to /content/drive/MyDrive/CPI-Analysis-Results/logit-lens-v2


In [ ]:
import json, yaml, torch, sys, shutil, time
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

sys.path.insert(0, "src/evaluation")
import eval as cpi_eval

cfg = yaml.safe_load(open("src/mechanistic-analysis/checkpoints.yaml"))
tok = AutoTokenizer.from_pretrained(cfg["base_model"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Llama-2-7b-hf ships no chat template; must match run_sft.py's training format exactly.
tok.chat_template = (
    "{{ bos_token }} "
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "{{ '<|system|>\n' + message['content'].strip() + '\n' }}"
    "{% elif message['role'] == 'user' %}"
    "{{ '<|user|>\n' + message['content'].strip() + '\n' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ '<|assistant|>\n' + message['content'].strip() + eos_token + '\n' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|assistant|>\n' }}{% endif %}"
)

# the prompt must match what the stored evals used, byte for byte
ref = json.load(open("results/cpi-results/tulu-results/checkpoint_step3150_metrics.json", encoding="utf-8"))
row = next(r for r in ref["per_item"] if r["item_id"] == "cap_0013")
ours = tok.decode(cpi_eval.build_score_prompt_ids(tok, row["question"], row["context"], True),
                  skip_special_tokens=False)
assert ours == row["score_prompt_text"], f"PROMPT MISMATCH\nours  : {ours!r}\nstored: {row['score_prompt_text']!r}"
print("prompt verified")

items = cpi_eval.load_items("dataset/conflict_eval_unified.json")
base_model = AutoModelForCausalLM.from_pretrained(
    cfg["base_model"], torch_dtype=torch.bfloat16, device_map="cuda").eval()
print(len(items), "items; base model loaded")

prompt verified
Loaded 417 total | dropped 5 flagged item(s) | 412 usable


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

412 items; base model loaded


In [ ]:
# Which checkpoints to read. Seed 0 comes from checkpoints.yaml.
# To add another seed: put its Drive checkpoint folder in EXTRA_SEEDS. Its phase steps default to
# seed 0's (you recalled: same trough, peak within +-100 steps); change them here if you know better.
EXTRA_SEEDS = {
    # ("tulu", 1):   "/content/drive/MyDrive/<folder with checkpoint-500, checkpoint-3150, ...>",
    # ("alpaca", 1): "/content/drive/MyDrive/<folder with checkpoint-100, checkpoint-700>",
}

def phase_steps(run):
    return {p: int(v.split("-")[1]) for p, v in cfg["runs"][run]["phases"].items() if v}

folders = {(run, 0): cfg["runs"][run]["drive_checkpoint_dir"] for run in ["tulu", "alpaca"]}
folders.update(EXTRA_SEEDS)

CHECKPOINTS = []
for (run, seed), folder in folders.items():
    for phase, step in phase_steps(run).items():
        path = f"{folder}/checkpoint-{step}"
        assert os.path.isdir(path), f"missing checkpoint: {path}"
        CHECKPOINTS.append({"run": run, "seed": seed, "phase": phase, "step": step, "path": path})

for c in CHECKPOINTS:
    print(c["run"], "seed", c["seed"], c["phase"], c["step"])

tulu seed 0 peak 500
tulu seed 0 trough 3150
tulu seed 0 recovery 5000
alpaca seed 0 peak 100
alpaca seed 0 trough 700


In [ ]:
STAGE = "/content/_ckpt"

def load(adapter_path):
    # base model with the given LoRA adapter attached
    global base_model
    if isinstance(base_model, PeftModel):
        base_model = base_model.unload()
    shutil.rmtree(STAGE, ignore_errors=True)
    shutil.copytree(adapter_path, STAGE)          # local copy: Drive reads are slow
    base_model = PeftModel.from_pretrained(base_model, STAGE).eval()
    return base_model

NEWLINE = tok("\n", add_special_tokens=False).input_ids[-1]      # the <0x0A> byte token

def first_token(answer):
    # The generation prompt ends in a newline, so the answer is tokenised as it would follow one:
    # tokenise "\n" + answer and take the token right after the newline.
    ids = tok("\n" + answer.strip(), add_special_tokens=False).input_ids
    return ids[ids.index(NEWLINE) + 1]

@torch.no_grad()
def lens_curve(model, item, return_top=False):
    # D[layer] for layer 0 (embeddings) .. 32 (final output), at the last prompt position
    ids = cpi_eval.build_gen_prompt_ids(tok, item["question"], item["context"], True)
    out = model(ids.unsqueeze(0).to("cuda"), output_hidden_states=True)
    llama = model.get_base_model()
    hs = out.hidden_states
    assert len(hs) == llama.config.num_hidden_layers + 1

    # Layers before the last: raw residual stream -> final norm -> unembedding (the "lens").
    # Last layer: the model's real output logits. (transformers versions differ on whether
    # hidden_states[-1] is already normalised, so we don't rely on it.)
    logits = [llama.lm_head(llama.model.norm(h[0, -1])) for h in hs[:-1]] + [out.logits[0, -1]]
    logits = torch.stack(logits).float()

    c, p = first_token(item["counterfactual_answer"]), first_token(item["parametric_answer"])
    D = [round(x, 3) for x in (logits[:, c] - logits[:, p]).tolist()]
    if return_top:
        return D, logits[-1].argmax().item()        # the token the model would actually generate
    return D

In [ ]:
# Sanity check before the full run, on seed-0 tulu peak, against the stored greedy generations.
# Same prompt + greedy decoding, so the final layer's top token should be exactly the first token
# the eval generated. For items whose generation starts with one of the two answers:
#   (1) top token == first_token(that answer)   -> prompt and tokenisation are right
#   (2) sign of D[-1] points to that answer      -> the lean measures the behaviour
first = next(c for c in CHECKPOINTS if c["run"] == "tulu" and c["seed"] == 0 and c["phase"] == "peak")
stored = {r["item_id"]: r for r in json.load(open(
    f"results/cpi-results/tulu-results/checkpoint_step{first['step']}_metrics.json", encoding="utf-8"))["per_item"]}

model = load(first["path"])
n = top_ok = sign_ok = 0
for it in items[:80]:
    par, cf = it["parametric_answer"].strip(), it["counterfactual_answer"].strip()
    if first_token(par) == first_token(cf):
        continue
    resp = stored[it["item_id"]]["response"].strip()
    said = "cf" if resp.startswith(cf) else ("par" if resp.startswith(par) else None)
    if said is None:
        continue
    D, top = lens_curve(model, it, return_top=True)
    n += 1
    top_ok += top == first_token(cf if said == "cf" else par)
    sign_ok += (D[-1] > 0) == (said == "cf")
    if n <= 3:
        print(f"{it['item_id']}: generated {resp[:25]!r}  top token {tok.decode([top])!r}  D every 4th layer: {D[::4]}")
print(f"(1) top token = first token of the generated answer: {top_ok}/{n}  (want ~all)")
print(f"(2) lean points to the generated answer:           {sign_ok}/{n}  (want ~all)")

cap_0000: generated 'Herat</s>\n\nAnswer: Herat<'  top token 'Her'  D every 4th layer: [0.638, -0.761, -1.492, -2.164, -0.848, 0.5, 0.562, 6.375, 1.75]
cap_0001: generated 'Durrës</s>\n\nAnswer: Durrë'  top token 'D'  D every 4th layer: [1.148, 1.234, 3.26, -0.188, 0.352, 3.531, -1.312, 0.562, 0.375]
cap_0002: generated 'Oran</s>\n\nAnswer: Oran</s'  top token 'O'  D every 4th layer: [-2.592, 0.107, -0.883, 2.696, 0.584, 5.5, 1.469, 5.219, 1.5]
(1) top token = first token of the generated answer: 74/75  (want ~all)
(2) lean points to the generated answer:           74/75  (want ~all)


In [ ]:
# Full run. Resumable: a checkpoint whose JSON is already on Drive is skipped.
for c in CHECKPOINTS:
    out_file = f"{OUT_DIR}/{c['run']}_seed{c['seed']}_{c['phase']}_step{c['step']}.json"
    if os.path.exists(out_file):
        print("skip (done):", out_file)
        continue
    t0 = time.time()
    model = load(c["path"])
    curves = {}
    for it in items:
        curves[it["item_id"]] = {
            "D": lens_curve(model, it),
            "same_first_token": first_token(it["parametric_answer"]) == first_token(it["counterfactual_answer"]),
        }
    json.dump({**c, "curves": curves}, open(out_file, "w"))
    print(f"{c['run']} seed{c['seed']} {c['phase']}: {len(curves)} items in {time.time() - t0:.0f}s")

shutil.rmtree(STAGE, ignore_errors=True)
print("done -- download the folder", OUT_DIR, "to src/mechanistic-analysis/logit-lens-v2/")

tulu seed0 peak: 412 items in 44s
tulu seed0 trough: 412 items in 44s
tulu seed0 recovery: 412 items in 50s
alpaca seed0 peak: 412 items in 44s
alpaca seed0 trough: 412 items in 45s
done -- download the folder /content/drive/MyDrive/CPI-Analysis-Results/logit-lens-v2 to src/mechanistic-analysis/logit-lens-v2/
